In [0]:
!rm -rf *

In [0]:
from __future__ import print_function
import os
import json
import six.moves.urllib as urllib
import tarfile
import zipfile
import numpy as np
def exec_cmd(cmdstr,echo=True):
  print(os.popen(cmdstr).read() if echo else '',end='')

def download_file(url,filename=None,nc=False):#nc no clobber i.e dont download if file exists
  if nc:
    fname = filename if filename is not None else  url.split('/')[-1]
    if os.path.isfile(fname):
      print('File ',fname,'exists. Skipping download')
      return fname
  print('Downloading from',url)
  fn = (str(' -O '+filename) if filename else ' ')
  exec_cmd('wget -nc '+url+fn)
  return filename if filename else url.split('/')[-1]

def download_file_urllib(url,filename):
  print('Downloading from',url,' to ',filename)
  opener = urllib.request.URLopener()
  opener.retrieve(url, filename)
  print('Downloaded file size: ',os.stat(filename).st_size/1048576,'Mb')

def create_zip(zip_name,file_names):
  print('Creating zip file ',zip_name)
  exec_cmd('zip -r '+zip_name+' '+' '.join(file_names))

def extract_file_from_tar(tar_file,filename_to_ext,dest_dir):
  print('Trying to extract',filename_to_ext,'from',tar_file)
  tar_file = tarfile.open(tar_file)
  for file in tar_file.getmembers():
    file_name = os.path.basename(file.name)
    if filename_to_ext in file_name:
      tar_file.extract(file, dest_dir)
      
def load_image_into_numpy_array(self,image):
  (im_width, im_height) = image.size
  return np.array(image.getdata()).reshape((im_height, im_width, 3)).astype(np.uint8)
 
def unannotate_image(ann_dict): #once per file
  #annotation format to gof - general output format the way yolo/ssd give their o/p
  #output_dict = {'output':{fname:{}}, 'labels_matched':None, 'time':-1} #dict in gof
  fname = ann_dict['annotation']['data_filename']
  bbox_anns = ann_dict['annotation']['data_annotation']['bounding_box']
  bbox_annotations = {}
  bbk,lbk = 'bounding_boxes','labels_detected'
  bbox_annotations[bbk] = []
  bbox_annotations[lbk] = []
  bbk_lst,lbk_lst  = bbox_annotations[bbk], bbox_annotations[lbk]
  for ba in bbox_anns:
    lbk_lst.append(ba['classification_label'])
    coords =  list(map(lambda x:  tuple(map(lambda v: int(v), x.split(','))),ba['point_2D'] )) # "5,6","3,4" = > ((5,6),(3,4))
    bbk_lst.append(list( coords[0]+coords[1] ))
  return  {fname:{'labels_matched': [], 'output': {bbk:bbk_lst,lbk:lbk_lst} , 'time':-1}}

def unannotate_all(annotated_output):
  opd = {}
  for d in list(map(unannotate_image,annotated_output)):
    opd.update(d)
  return opd
  
import json
def bbox_annotations(output): # call once per file
  bbox_annotations=[]
  bbk,lbk = 'bounding_boxes','labels_detected'
  tmp = output['output']
  for i in range(len(tmp[bbk])):
    coordinates = [str(val) for val in tmp[bbk][i]]
    pt1 = ','.join(coordinates[:2])
    pt2 = ','.join(coordinates[2:])
    bbox_annotations.append({'classification_label':tmp[lbk][i],'point_2D':[pt1,pt2]})
  return bbox_annotations

def annotate_image(output_item):  # call once per file
  fname, output_dict = output_item
  annotations = {'annotation': {'data_filename': fname, 'data_type': 'image', 'data_annotation': {'bounding_polygon': [], 'bounding_box': ''}}}
  annotations['annotation']['data_annotation']['bounding_box'] = bbox_annotations(output_dict)
  return annotations

def save_as_json(dct,parent_dir='./'): # call once per file
  fname = parent_dir+dct['annotation']['data_filename']
  fname = '.'.join(fname.split('.')[:-1])+'.json'
  try:
    with open(fname,'w+') as f:
      json.dump(dct,f)
    return fname
  except Exception as e:
    print('Error writing to '+fname,'\n',e)
  return ('Failed!',fname)
  
def save_as_annotations(output,opdir='./'):
  print('Saving annotations to file')
  opdir = opdir if opdir[-1]=='/' else opdir+'/' #add trailing / if not present
  annotated_output = list(map(annotate_image,list(output.items())))
  success,failure=[],[]
  for dct in annotated_output:
    fname = save_as_json(dct,opdir)
    lst = failure if type(fname) is tuple else success
    lst.append(fname)
  return success,failure,annotated_output
  #result = [{dct['annotation']['data_filename']:save_as_json(dct)} for dct in annotated_output ]
  #return result,list(filter(lambda x: type(tuple(x.items())[0][-1]) is tuple ,gg))

In [0]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals
import gzip
import io
from collections import defaultdict
import argparse
import cv2  # NOQA (Must import before importing caffe2 due to bug in cv2)
import glob
import logging
import os
import sys
import time
from caffe2.python import workspace
import re


class detectron_fb:
  def __init__(self,model_name='RetinaNet',env_parent='models/',weights_url=None):
    self.model_name = model_name
    self.env_parent = ''#env_parent
    self.env = str(self.env_parent+'env/')
    self.env = self.env
    self.name = 'Detectron'
    self.cocoapi = self.env+'cocoapi'
    self.env_dir = self.env+self.name
    self.pretrained_models_dir = self.env+'pretrained/'
    self.prepared = False
    weights_url = weights_url if weights_url else 'https://dl.fbaipublicfiles.com/detectron/36768744/12_2017_baselines/retinanet_R-101-FPN_1x.yaml.08_31_38.5poQe1ZB/output/train/coco_2014_train%3Acoco_2014_valminusminival/retinanet/model_final.pkl'
    weights_url = str(weights_url)
    self.model = {'weights_url': weights_url}
    
    self.model['cfg'] = self.env_dir+'/configs/'+re.search(r'/[0-9]+/(.*yaml)',self.model['weights_url']).groups()[0] #12_2017_baselines/retinanet_X-101-32x8d-FPN_2x.yaml'
    self.model['weights_file']='.'.join(self.model['cfg'].split('/')[-1].split('.')[:-1])+'.pkl'
    self.model['weights']=self.pretrained_models_dir+self.model['weights_file']
    
    
  def download_weights(self):
    download_weights = True
    if os.path.isfile(self.model['weights']) and (os.stat(self.model['weights']).st_size/1048576)>1:
      print('weights file exists. size: ',(os.stat(self.model['weights']).st_size/1048576),'Mb')
      download_weights = False
    if download_weights:
      print('Downloadng weights:')#,self.model['weights_url'],'=>',self.model['weights'])
      download_file_urllib(self.model['weights_url'],self.model['weights'])
      
  def prepare_env(self):
    exec_cmd('mkdir -p '+self.env)
    exec_cmd('git clone https://github.com/cocodataset/cocoapi.git '+self.cocoapi)
    exec_cmd('cd '+self.cocoapi+'/PythonAPI && make install')

    exec_cmd('git clone https://github.com/facebookresearch/detectron '+self.env_dir)
    exec_cmd('ln -s '+self.cocoapi+' '+self.env_dir+'cocoapi')
    exec_cmd('pip install -r '+self.env_dir+'/requirements.txt')
    exec_cmd('cd '+self.env_dir+' && make')
    exec_cmd('python '+self.env_dir+'/detectron/tests/test_spatial_narrow_as_op.py')
    exec_cmd('mkdir -p '+self.pretrained_models_dir)
    self.download_weights()
      
  def import_dependencies(self):
    global assert_and_infer_cfg,cfg,merge_cfg_from_file,cache_url,setup_logging,Timer,infer_engine,dummy_datasets,c2_utils,vis_utils
    try:
      #sys.path.append(str(os.getcwd()+'/'+self.env[:-1])) #remove the trailing /
      detdir = str(os.getcwd()+'/'+self.env_dir)
      if detdir not in sys.path:
        sys.path.append(detdir)
      from detectron.core.config import assert_and_infer_cfg
      from detectron.core.config import cfg
      from detectron.core.config import merge_cfg_from_file
      from detectron.utils.io import cache_url
      from detectron.utils.logging import setup_logging
      from detectron.utils.timer import Timer
      import detectron.core.test_engine as infer_engine
      import detectron.datasets.dummy_datasets as dummy_datasets
      import detectron.utils.c2 as c2_utils
      import detectron.utils.vis as vis_utils
      c2_utils.import_detectron_ops()
      # OpenCL may be enabled by default in OpenCV3; disable it because it's not
      # thread safe and causes unwanted GPU memory allocations.
      cv2.ocl.setUseOpenCL(False)
      
      print('Successfully imported all detectron dependencies')
      return True
    except Exception as e:
      print('Unable to import detectron dependencies\n',e)
    return False
  
  def prepare(self):
    prepared = self.import_dependencies()
    if not prepared:
      print('Preparing Environment')
      self.prepare_env()
      self.prepared =  self.import_dependencies()
    else:
      self.prepared = True
      print('No need to prepare environment')
    return self.prepared
  
  def load(self):
    if not self.prepared:
      print('NOT PREPARED. QUITTING')
      return
    self.download_weights()
    if not os.path.isfile(self.model['cfg']):
      print(self.model['cfg'],'Not found! . Quitting')
      return
    if not os.path.isfile(self.model['weights']):
      print(self.model['weights'],'Not found! . Quitting')
      return
    
    merge_cfg_from_file(self.model['cfg'])
    try:
      cfg.NUM_GPUS = 1
      assert_and_infer_cfg(cache_urls=False)
    except Exception as e:
      print('re setting global values may have caused a warning',e)
    
    assert not cfg.MODEL.RPN_ONLY,'RPN models are not supported'
    assert not cfg.TEST.PRECOMPUTED_PROPOSALS,'Models that require precomputed proposals are not supported'

    self.model_obj = infer_engine.initialize_model_from_cfg(self.model['weights'])
    self.dataset = dummy_datasets.get_coco_dataset()
    
    
  def detect_(self,cvimg,opfname='predictions.jpg',visualize=False,threshold=0.7):
    if not self.prepared:
      print('NOT PREPARED. QUITTING')
      return
    
    with c2_utils.NamedCudaScope(0):
      cls_boxes, cls_segms, cls_keyps = infer_engine.im_detect_all(self.model_obj, cvimg, None)
    
    opdir = '/'.join(opfname.split('/')[:-1])
    opdir = '.' if not opdir else opdir
    opfname = opfname.split('/')[-1]
    opext = opfname.split('.')[-1]
    class_names,bboxes = [],[]
    if visualize:
      opfname = '.'.join(opfname.split('.')[:-1])
      vis_utils.vis_one_image( cvimg[:, :, ::-1],  # BGR -> RGB for visualization
            opfname,
            opdir,
            cls_boxes,
            cls_segms,
            cls_keyps,
            dataset=self.dataset,
            box_alpha=0.3,
            show_class=True,
            thresh=threshold,
            kp_thresh=threshold,
            ext=opext,
            out_when_no_box=True
        )
    else:
      opdir = opdir+'/' if opdir[-1]!='/' else opdir
      cv2.imwrite(opdir+opfname,cvimg)
    boxes, segms, keypoints, class_ids = vis_utils.convert_from_cls_format(cls_boxes, cls_segms, cls_keyps)
      
    if segms is not None and len(segms) > 0:
        masks = mask_util.decode(segms)
        color_list = colormap()
        mask_color_id = 0
    # sort in order of largest to smallest order to reduce occlusion
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    sorted_inds = np.argsort(-areas)
    for i in sorted_inds:
      bbox = list(boxes[i, :4])
      score = boxes[i, -1]
      if score < threshold:
          continue
      class_name = str(self.dataset.classes[class_ids[i]])
      class_names.append(class_name)
      bboxes.append(bbox)
        
    return class_names,bboxes

  def detect(self,image,opfile,class_labels_to_filter,visualize=False):
    opfile = opfile if opfile else 'detections.jpg'
    result = self.detect_(image,opfile,visualize=visualize)
    if not result:
      result = [],[]
    labels_detected, bboxes = result
    labels_matched = [ str(x) for x in list(set(labels_detected)&set(class_labels_to_filter))]
    output_dict = {str('bounding_boxes'):bboxes,str('labels_detected'):labels_detected}
    return {str('labels_matched'): labels_matched, str('output'): output_dict}


In [8]:
d = detectron_fb()
d.prepare()
d.load()

Unable to import detectron dependencies
 No module named detectron.core.config
Preparing Environment
# install pycocotools to the Python site-packages
python setup.py build_ext install
running build_ext
skipping 'pycocotools/_mask.c' Cython extension (up-to-date)
building 'pycocotools._mask' extension
creating build
creating build/common
creating build/temp.linux-x86_64-2.7
creating build/temp.linux-x86_64-2.7/pycocotools
x86_64-linux-gnu-gcc -pthread -DNDEBUG -g -fwrapv -O2 -Wall -Wstrict-prototypes -fno-strict-aliasing -Wdate-time -D_FORTIFY_SOURCE=2 -g -fdebug-prefix-map=/build/python2.7-3hk45v/python2.7-2.7.15~rc1=. -fstack-protector-strong -Wformat -Werror=format-security -fPIC -I/usr/local/lib/python2.7/dist-packages/numpy/core/include -I../common -I/usr/include/python2.7 -c ../common/maskApi.c -o build/temp.linux-x86_64-2.7/../common/maskApi.o -Wno-cpp -Wno-unused-function -std=c99
x86_64-linux-gnu-gcc -pthread -DNDEBUG -g -fwrapv -O2 -Wall -Wstrict-prototypes -fno-strict-aliasi

In [9]:
d.detect(cv2.imread(d.env_dir+'/demo/17790319373_bd19b24cfc_k.jpg'),'pr.jpg',['person'])

NOT PREPARED. QUITTING


{'labels_matched': [], 'output': {'bounding_boxes': [], 'labels_detected': []}}

In [10]:
sys.path

['',
 '/env/python',
 '/usr/lib/python2.7',
 '/usr/lib/python2.7/plat-x86_64-linux-gnu',
 '/usr/lib/python2.7/lib-tk',
 '/usr/lib/python2.7/lib-old',
 '/usr/lib/python2.7/lib-dynload',
 '/root/.local/lib/python2.7/site-packages',
 '/usr/local/lib/python2.7/dist-packages',
 '/usr/local/lib/python2.7/dist-packages/pycocotools-2.0-py2.7-linux-x86_64.egg',
 '/usr/local/lib/python2.7/dist-packages',
 '/usr/lib/python2.7/dist-packages',
 '/usr/local/lib/python2.7/dist-packages/IPython/extensions',
 '/content/env/Detectron',
 '/root/.ipython']